In [2]:
import re
import os
import ast
import matplotlib
from dataclasses import dataclass, field
from collections import defaultdict
MATCH_PARA = re.compile(r'(\w+)\s+([\d.]+)')
import matplotlib.pyplot as plt
import math
from Utils import *
data_path = "quantum_network_compiler-minimal_codes/FIG_dataset_rebuttal"
current_dir = os.getcwd()
parent_dir = os.path.dirname(os.path.dirname(current_dir))
absolute_path = os.path.join(parent_dir, data_path)
data_path = str(os.path.abspath(absolute_path))
data_path_to_execute = data_path + '/qpu_per_rack'
benchmark = ['xor', 'qft','grover','rca', 'qaoa']
eval = 'qpu_per_rack'
big_list = process_files_by_seed(data_path_to_execute)
averages = calculate_cross_sublist_averages(big_list)
result = classify_and_sort_averages_1(averages,eval)
result_base = classify_and_sort_averages_1(averages, eval, our=1)
diction_our = [[[],[],[],[], []],
               [[],[],[],[], []],
               [[],[],[],[], []],
               [[],[],[],[], []],
               [[],[],[],[], []],
               [[],[],[],[], []],
               [[],[],[],[], []]]

diction_baseline = [[[],[],[],[], []],
                    [[],[],[],[], []],
                    [[],[],[],[], []],
                    [[],[],[],[], []],
                    [[],[],[],[], []],
                    [[],[],[],[], []],
                    [[],[],[],[], []]]

# diction_our = [[[],[],[],[],],
#                [[],[],[],[], ],
#                [[],[],[],[], ],
#                [[],[],[],[], ],
#                [[],[],[],[], ],
#                [[],[],[],[], ],
#                [[],[],[],[], ]]
#
# diction_baseline = [[[],[],[],[], ],
#                     [[],[],[],[], ],
#                     [[],[],[],[], ],
#                     [[],[],[],[], ],
#                     [[],[],[],[], ],
#                     [[],[],[],[], ],
#                     [[],[],[],[], ]]

for index, b in enumerate(benchmark):
    for a in result[b]:
        diction_our[0][index].append(a['avg_metric_t']) #throughput
        diction_our[1][index].append(a['avg_metric_w']) # waittime
        diction_our[2][index].append(a['eproverhead']) #epr
        diction_our[3][index].append(a['avg_retry_overhead'])
        diction_our[4][index].append(a['cross'])
        diction_our[5][index].append(a['in_pair'])
        diction_our[6][index].append(a['distill'])

    for aa in result_base[b]:
        diction_baseline[0][index].append(aa['avg_metric_t'])
        diction_baseline[1][index].append(aa['avg_metric_w'])
        diction_baseline[2][index].append(aa['eproverhead'])
        diction_baseline[3][index].append(aa['avg_retry_overhead'])
        diction_baseline[4][index].append(aa['cross'])
        diction_baseline[5][index].append(aa['in_pair'])
        diction_baseline[6][index].append(aa['distill'])

def format_k(val):
    if val >= 1000:
        return f"{val/1000:,.0f}k"
    else:
        return f"{val:,.0f}"

def safe_ratio(our, base):
    if base == 0:
        return "N/A"
    return ((our - base) / our)

j = 0
benchmark = ['MCT','QFT','Grover','RCA', 'QAOA']
c_type = [240,360,480,600,720]
for index,_ in enumerate(diction_our[0]):
    for j in [1,2,3,4]:
        print(f'&{benchmark[index]}-{c_type[j]}'
        f'&{diction_baseline[0][index][j]:,.0f}'
        f'&{diction_our[0][index][j]:,.0f}'
        f'&$\\boldsymbol{{{diction_baseline[0][index][j]/diction_our[0][index][j]:.2f}\\times}}$'
        f'&{diction_baseline[4][index][j]:,.0f}'
        f'&{diction_our[5][index][j]:,.0f}'
        f'&{diction_our[6][index][j]:,.0f}'
        # f'&{((diction_our[2][index][j] - diction_baseline[2][index][j]) / diction_our[2][index][j]) * 100:,.2f}\\%'
        #f'&{safe_ratio(diction_our[2][index][j], diction_baseline[2][index][j]) * 100:,.2f}\\%'
        f'&{diction_baseline[1][index][j] / 10:,.2f}'
        f'&{diction_our[1][index][j] / 10:,.2f}'
        f'&{abs(diction_baseline[1][index][j] / 10 - diction_our[1][index][j] / 10):,.2f}'
        f'&{diction_our[3][index][j]:,.2f}\\\\')
        print('\\cline{2-13}')
# throughput-baseline-our-improv ; epr-baseline-our-overhead ; waittime-baseline-our-delta


&MCT-360&1,476&468&$\boldsymbol{3.15\times}$&15&144&15&3.03&5.81&2.78&0.00\\
\cline{2-13}
&MCT-480&2,312&485&$\boldsymbol{4.77\times}$&15&240&13&1.98&6.75&4.77&0.00\\
\cline{2-13}
&MCT-600&3,214&634&$\boldsymbol{5.07\times}$&15&432&12&1.17&5.30&4.13&0.00\\
\cline{2-13}
&MCT-720&4,413&921&$\boldsymbol{4.79\times}$&15&624&11&0.87&5.27&4.40&0.00\\
\cline{2-13}
&QFT-360&78,300&13,504&$\boldsymbol{5.80\times}$&810&1,500&715&1.48&6.27&4.79&0.00\\
\cline{2-13}
&QFT-480&121,728&16,693&$\boldsymbol{7.29\times}$&1,080&2,970&1,133&1.30&6.87&5.57&0.00\\
\cline{2-13}
&QFT-600&169,831&20,041&$\boldsymbol{8.47\times}$&1,350&4,920&1,559&1.14&7.00&5.86&0.00\\
\cline{2-13}
&QFT-720&216,372&23,362&$\boldsymbol{9.26\times}$&1,620&7,350&1,915&1.20&7.63&6.43&0.00\\
\cline{2-13}
&Grover-360&140,813&29,717&$\boldsymbol{4.74\times}$&1,800&4,800&1,594&2.03&9.96&7.93&0.00\\
\cline{2-13}
&Grover-480&156,213&26,943&$\boldsymbol{5.80\times}$&1,800&7,200&1,927&2.08&8.73&6.65&0.00\\
\cline{2-13}
&Grover-600&171,613&2

In [4]:
import re
import os
import ast
import matplotlib
from dataclasses import dataclass, field
from collections import defaultdict
MATCH_PARA = re.compile(r'(\w+)\s+([\d.]+)')
import matplotlib.pyplot as plt
import math
from Utils import *
data_path = "quantum_network_compiler-minimal_codes/FIG_dataset_rebuttal"
current_dir = os.getcwd()
parent_dir = os.path.dirname(os.path.dirname(current_dir))
absolute_path = os.path.join(parent_dir, data_path)
data_path = str(os.path.abspath(absolute_path))
data_path_to_execute = data_path + '/qpu_per_rack'
benchmark = ['xor', 'qft', 'grover', 'rca', 'qaoa']
eval = 'qpu_per_rack'
big_list = process_files_by_seed(data_path_to_execute)
averages = calculate_cross_sublist_averages(big_list)
result = classify_and_sort_averages_1(averages,eval)
result_base = classify_and_sort_averages_1(averages, eval, our=1)
diction_our = [[[],[],[],[],[]],
               [[],[],[],[],[]],
               [[],[],[],[],[]],
               [[],[],[],[],[]],
               [[],[],[],[],[]],
               [[],[],[],[],[]],
               [[],[],[],[],[]]]
diction_baseline = [[[],[],[],[],[]],
                    [[],[],[],[],[]],
                    [[],[],[],[],[]],
                    [[],[],[],[],[]],
                    [[],[],[],[],[]],
                    [[],[],[],[],[]],
                    [[],[],[],[],[]]]
for index, b in enumerate(benchmark):
    for a in result[b]:
        diction_our[0][index].append(a['avg_metric_t']) #throughput
        diction_our[1][index].append(a['avg_metric_w']) # waittime
        diction_our[2][index].append(a['eproverhead']) #epr
        diction_our[3][index].append(a['avg_retry_overhead'])
        diction_our[4][index].append(a['cross'])
        diction_our[5][index].append(a['in_pair'])
        diction_our[6][index].append(a['distill'])

    for aa in result_base[b]:
        diction_baseline[0][index].append(aa['avg_metric_t'])
        diction_baseline[1][index].append(aa['avg_metric_w'])
        diction_baseline[2][index].append(aa['eproverhead'])
        diction_baseline[3][index].append(aa['avg_retry_overhead'])
        diction_baseline[4][index].append(aa['cross'])
        diction_baseline[5][index].append(aa['in_pair'])
        diction_baseline[6][index].append(aa['distill'])

def format_k(val):
    if val >= 1000:
        return f"{val/1000:,.0f}k"
    else:
        return f"{val:,.0f}"

j = 0
benchmark = ['MCT','QFT','Grover','RCA','QAOA']
c_type = [240,360,480,600,'720$^*$']
for index,_ in enumerate(diction_our[0]):
    for j in [1,2,3,4]:
        print(f'&{benchmark[index]}-{c_type[j]}'
        f'&{diction_baseline[0][index][j]:,.0f}'
        f'&{diction_our[0][index][j]:,.0f}'
        f'&$\\boldsymbol{{{diction_baseline[0][index][j]/diction_our[0][index][j]:.2f}\\times}}$'
        f'&{diction_baseline[4][index][j]:,.0f}'
        f'&{diction_our[5][index][j]:,.0f}'
        f'&{diction_our[6][index][j]:,.0f}'

            # f'&{((diction_our[2][index][j] - diction_baseline[2][index][j]) / diction_our[2][index][j]) * 100:,.2f}\\%'
        f'&{diction_baseline[1][index][j] / 10:,.2f}'
        f'&{diction_our[1][index][j] / 10:,.2f}'
        f'&{abs(diction_baseline[1][index][j] / 10 - diction_our[1][index][j] / 10):,.2f}'
        f'&{diction_our[3][index][j]:,.2f}\\\\')
        print('\\cline{2-13}')
# throughput-baseline-our-improv ; epr-baseline-our-overhead ; waittime-baseline-our-delta


&MCT-360&1,476&468&$\boldsymbol{3.15\times}$&15&144&15&3.03&5.81&2.78&0.00\\
\cline{2-13}
&MCT-480&2,312&485&$\boldsymbol{4.77\times}$&15&240&13&1.98&6.75&4.77&0.00\\
\cline{2-13}
&MCT-600&3,214&634&$\boldsymbol{5.07\times}$&15&432&12&1.17&5.30&4.13&0.00\\
\cline{2-13}
&MCT-720$^*$&4,413&921&$\boldsymbol{4.79\times}$&15&624&11&0.87&5.27&4.40&0.00\\
\cline{2-13}
&QFT-360&78,300&13,504&$\boldsymbol{5.80\times}$&810&1,500&715&1.48&6.27&4.79&0.00\\
\cline{2-13}
&QFT-480&121,728&16,693&$\boldsymbol{7.29\times}$&1,080&2,970&1,133&1.30&6.87&5.57&0.00\\
\cline{2-13}
&QFT-600&169,831&20,041&$\boldsymbol{8.47\times}$&1,350&4,920&1,559&1.14&7.00&5.86&0.00\\
\cline{2-13}
&QFT-720$^*$&216,372&23,362&$\boldsymbol{9.26\times}$&1,620&7,350&1,915&1.20&7.63&6.43&0.00\\
\cline{2-13}
&Grover-360&140,813&29,717&$\boldsymbol{4.74\times}$&1,800&4,800&1,594&2.03&9.96&7.93&0.00\\
\cline{2-13}
&Grover-480&156,213&26,943&$\boldsymbol{5.80\times}$&1,800&7,200&1,927&2.08&8.73&6.65&0.00\\
\cline{2-13}
&Grover-600&1